In [1]:
from shopping_assistant.env import load_env
load_env()

True

In [9]:
from openai import OpenAI
from agents import function_tool, Agent, Runner, set_tracing_disabled
import agents

set_tracing_disabled(True)

openai_client = OpenAI()

In [10]:
from typing import Annotated
from collections import Counter

class Cart:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.items = Counter()

    def add_item(self, item, quantity):
        self.items[item] += quantity
        return self

    def remove_item(self, item, quantity):
        cur_qty = self.items.get(item, 0)
        if cur_qty < quantity:
            raise ValueError("Not enough quantity in cart")
        if cur_qty == quantity:
            return self.empty_item(item)
    
        self.items[item] -= quantity
        return self

    def empty_cart(self):
        self.items.clear()
        return self

    def empty_item(self, item):
        self.items.pop(item, None)
        return self

    def view_cart(self):
        return dict(self.items)


def get_cart_action_tools_for_user(cart: Cart) -> list:

    @function_tool
    def add_to_cart(
        product_id: Annotated[str, "product_id"],
        quantity: Annotated[int, "quantity"] = 1
    ) -> dict:
        """Add a product to user's cart"""
        try:
            cart.add_item(product_id, quantity)
        except Exception as e:
            return "Error updating cart: " + str(e)
        
        return f"Cart updated successfully. Current Cart: {cart.view_cart()}"

    @function_tool
    def remove_from_cart(product_id: str, quantity: Annotated[int, "quantity"]) -> dict:
        """Remove a product from user's cart. Returns the updated cart."""
        cart.remove_item(product_id, quantity)
        return f"Cart updated successfully. Current Cart: {cart.view_cart()}"

    @function_tool
    def view_cart() -> list:
        """View user's cart"""
        return cart.view_cart()

    return [
        add_to_cart,
        remove_from_cart,
        view_cart
    ]




In [11]:
cart = Cart('123')

print(cart.view_cart())

cart.add_item('item_1', quantity=1)
print(cart.view_cart())

cart.remove_item('item_1', quantity=1)
print(cart.view_cart())



{}
{'item_1': 1}
{}


In [12]:
cart = Cart(user_id="123")

SHOPPING_AGENT_ACTIONS = {
    "cart": {
        "summary": "Manage your shopping cart (add or remove items, view cart)",
        "tools": get_cart_action_tools_for_user(cart),
    }
}

ACTION_SUMMARY = '\n'.join([
    f"{idx + 1}. Action=`{action_type}`: {info['summary']}"
    for idx, (action_type, info) in enumerate(SHOPPING_AGENT_ACTIONS.items())
])

ACTION_DETAILED = ''
for idx, (action_type, info) in enumerate(SHOPPING_AGENT_ACTIONS.items()):
    ACTION_DETAILED += f"{idx + 1}. Action=`{action_type}`: {info['summary']}:\n"
    for tool in info['tools']:
        ACTION_DETAILED += f"- `{tool.name}`: {tool.description}\n"

ACTION_TYPES = list(SHOPPING_AGENT_ACTIONS.keys())


SHOPPING_ACTION_TOOLS = [
    tool
    for action in SHOPPING_AGENT_ACTIONS.values()
    for tool in action['tools']
]

In [13]:
print(ACTION_DETAILED)

1. Action=`cart`: Manage your shopping cart (add or remove items, view cart):
- `add_to_cart`: Add a product to user's cart
- `remove_from_cart`: Remove a product from user's cart. Returns the updated cart.
- `view_cart`: View user's cart



In [14]:
SYSTEM_PROMPT = f"""
You are a helpful shopping assistant that can help users with their shopping needs.
Based on the user's query, you do the following:

Step 1. Parse the user's query to understand the user's intented action type out of the following list: {ACTION_TYPES})
Step 2. If no action type is identified, you respond verbatim with:

```
I'm sorry, I don't understand your query. I can help with the following actions:
{ACTION_SUMMARY}
```
Step 3. Based on the action type, you will call the appropriate action function from the following list:

{ACTION_DETAILED}

Step 4. After the action is completed, you will inform the user of the action result.

"""

In [15]:
shopping_action_agent = Agent(
    name="shopping-action-agent",
    instructions=SYSTEM_PROMPT,
    tools=get_cart_action_tools_for_user(Cart('123')),
    model="gpt-4o-mini",
)

response = await Runner.run(shopping_action_agent, input="Can I know what's in my cart?")

In [16]:
print(response.final_output)

Your cart is currently empty. If you need help adding items, just let me know!


In [17]:
response = await Runner.run(shopping_action_agent, input="Can you add this 2 of this product to my cart? (item_524)")
print(response.final_output)

I've added 2 of the product (item_524) to your cart. Your current cart now contains 2 of this item.


In [18]:
response.raw_responses[0].output

[ResponseFunctionToolCall(arguments='{"product_id":"item_524","quantity":2}', call_id='call_fYSdq1GgMNZG1tBnUa0mXdFc', name='add_to_cart', type='function_call', id='fc_0494200a8a5b474d0169247beb688481a3aef64af8d0b24138', status='completed')]

In [19]:
response = await Runner.run(shopping_action_agent, input="What's in my cart")
print(response.final_output)

You have 2 units of item 524 in your cart.


In [20]:
response.raw_responses[0].output

[ResponseFunctionToolCall(arguments='{}', call_id='call_7kro2D0qBG4a0RxRIBLSBamF', name='view_cart', type='function_call', id='fc_0e00faf888f7e8880169247bf1393c81a3a1a70d83e8c13496', status='completed')]

In [21]:
response = await Runner.run(shopping_action_agent, input="Can you remove item_161 from my cart and tell me what's left?")
print(response.final_output)

I've removed item_161 from your cart. Currently, you have:

- item_524: 2 units

Is there anything else you need?


In [22]:
response.raw_responses

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"product_id":"item_161","quantity":1}', call_id='call_AyyVVySt6qCMpgUjzunTp4Q6', name='remove_from_cart', type='function_call', id='fc_01762342598d6fbc0169247bf69d9c819c97e102e33f233f47', status='completed'), ResponseFunctionToolCall(arguments='{}', call_id='call_GpJ0hF7miedojAFBFSBt6nZC', name='view_cart', type='function_call', id='fc_01762342598d6fbc0169247bf712e8819c853a18bb7e6c1e4d', status='completed')], usage=Usage(requests=1, input_tokens=178, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=50, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=228, request_usage_entries=[]), response_id='resp_01762342598d6fbc0169247bf58e74819cbb4bbd90b8420cf5'),
 ModelResponse(output=[ResponseFunctionToolCall(arguments='{"product_id":"item_161","quantity":0}', call_id='call_whHndXDk3SRmoqu8tajYFZvT', name='remove_from_cart', type='function_call', id='fc_01762342598d6fbc0169247bf8a8cc819c

In [23]:
response = await Runner.run(shopping_action_agent, input="Add 2 x item_1212, 4 x item_122. Now what's in my cart?")
print(response.final_output)

I've added 2 of item_1212 and 4 of item_122 to your cart. 

Currently, your cart contains:
- 2 x item_524
- 2 x item_1212
- 4 x item_122


In [24]:
response = await Runner.run(shopping_action_agent, input="Remove all the items in the cart.")
print(response.final_output)

All items have been successfully removed from your cart. Your cart is now empty! If you need anything else, feel free to ask.


In [26]:
# Source: https://github.com/openai/openai-agents-python/issues/351

from agents import MessageOutputItem, HandoffOutputItem, ToolCallItem, ToolCallOutputItem, ItemHelpers


def print_agent_response(response):
    for new_item in response.new_items:
        agent_name = new_item.agent.name
        if isinstance(new_item, MessageOutputItem):
            (f"{agent_name}: {ItemHelpers.text_message_output(new_item)}")
        elif isinstance(new_item, HandoffOutputItem):
            print(
                "Handed off from {new_item.source_agent.name} to {new_item.target_agent.name}"
            )
        elif isinstance(new_item, ToolCallItem):
            print(f"{agent_name}: TOOL CALL {new_item.raw_item.name}({new_item.raw_item.arguments})")
        elif isinstance(new_item, ToolCallOutputItem):
            print(f"{agent_name}: TOOL CALL OUTPUT: {new_item.output}")
        else:
            print(f"{agent_name}: Skipping item: {new_item.__class__.__name__}")

In [27]:
shopping_action_agent = Agent(
    name="shopping-action-agent",
    instructions=SYSTEM_PROMPT,
    tools=get_cart_action_tools_for_user(Cart('123')),
    model="gpt-4o-mini",
)

async def chat(user_input: str, history) -> str:
    processed_history = [
        {k: v for k, v in d.items() if k in ['role', 'content']}
        for d in history
    ]
    input_messages = processed_history + [
        {'role': 'user', 'content': user_input}
    ]
    response = await Runner.run(shopping_action_agent, input_messages)

    print_agent_response(response)

    return response.final_output
    

In [ ]:
import gradio as gr

gr.ChatInterface(chat, type="messages").launch()

/Users/Abhishek_Bhatia-GUVA/personal/projects/ecom-shopping-assistant/genai-shopping-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


shopping-action-agent: TOOL CALL view_cart({})
shopping-action-agent: TOOL CALL OUTPUT: {}
shopping-action-agent: TOOL CALL add_to_cart({"product_id":"item_213123","quantity":1})
shopping-action-agent: TOOL CALL OUTPUT: Cart updated successfully. Current Cart: {'item_213123': 1}
shopping-action-agent: TOOL CALL add_to_cart({"product_id":"item_213123","quantity":3})
shopping-action-agent: TOOL CALL add_to_cart({"product_id":"item_12312231","quantity":5})
shopping-action-agent: TOOL CALL OUTPUT: Cart updated successfully. Current Cart: {'item_213123': 4}
shopping-action-agent: TOOL CALL OUTPUT: Cart updated successfully. Current Cart: {'item_213123': 4, 'item_12312231': 5}
shopping-action-agent: TOOL CALL remove_from_cart({"product_id":"item_213123","quantity":4})
shopping-action-agent: TOOL CALL OUTPUT: Cart updated successfully. Current Cart: {'item_12312231': 5}
shopping-action-agent: TOOL CALL remove_from_cart({"product_id":"item_12312231","quantity":5})
shopping-action-agent: TOOL C